# Arheologia Mașinilor Inteligente: De la Kantorovich (1939) la Python

**Material Didactic** | **Durata:** 45 min  
**Subiect:** Programare Liniară, Optimizare de Resurse, Istoria Algoritmilor

---

## 1. Introducere și Context Istoric

În 1939, matematicianul sovietic **Leonid Kantorovich** a publicat lucrarea *"Metode matematice de organizare și planificare a producției"*. Într-o fabrică de placaj din Leningrad, el a observat că mașinile nu erau folosite eficient. Deși inginerii încercau să maximizeze producția, le lipsea o metodă matematică riguroasă pentru a aloca resursele.

Kantorovich a inventat ceea ce numim astăzi **Programare Liniară (Linear Programming)**, o metodă pentru a obține cel mai bun rezultat (profit maxim sau cost minim) într-un model matematic reprezentat prin relații liniare. Pentru această descoperire, a primit Premiul Nobel pentru Economie în 1975.

### Ce vom face astăzi?
1. Vom înțelege matematica din spatele **Problemei A** (Alocarea Mașinilor).
2. Vom folosi un solver modern (Python + PuLP) pentru a rezolva problema.
3. Vom compara rezultatele moderne cu cele calculate manual în 1939.



## 2. Modelul Matematic (Teorie)

Să presupunem că avem o fabrică cu mai multe tipuri de mașini și trebuie să producem seturi complete de piese (ex: un șurub și o piuliță formează un set).

### Variabilele
Notăm cu $h_{ik}$ fracțiunea de timp pe care mașina $i$ o petrece lucrând la piesa $k$.
* Dacă $h_{11} = 0.5$, înseamnă că Mașina 1 lucrează 50% din timp la Piesa 1.

### Funcția Obiectiv
Vrem să maximizăm numărul total de seturi complete, notat cu $Z$.
$$ \max Z $$

### Constrângerile (Regulile Jocului)

1. **Limita de Timp:** Fiecare mașină are o capacitate limitată (100% din timp). Dacă avem 3 mașini identice de tip $i$, capacitatea este 3.0.
   $$ \sum_{k} h_{ik} = \text{Număr Mașini}_i $$

2. **Balanța Producției:** Nu ne ajută să producem 1000 de șuruburi dacă avem doar 10 piulițe. Numărul de piese de tip $k$ produse trebuie să fie egal cu numărul total de seturi $Z$.
   $$ \sum_{i} (\text{Productivitate}_{ik} \times h_{ik}) = Z $$

3. **Non-negativitate:** Nu putem aloca timp negativ.
   $$ h_{ik} \ge 0 $$

In [ ]:
# Importam bibliotecile necesare
# PuLP este biblioteca standard in Python pentru Programare Liniara
try:
    import pulp
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    print("Biblioteci incarcate cu succes!")
except ImportError:
    print("Instalati bibliotecile necesare: pip install pulp numpy pandas seaborn matplotlib")

## 3. Motorul de Calcul (Solver)

Aceasta este clasa care transformă ecuațiile matematice de mai sus în cod Python. Folosim algoritmul **Simplex** (prin intermediul `PuLP`), care navighează prin colțurile poligonului de soluții posibile pentru a găsi optimul.

In [ ]:
import pulp
import numpy as np
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.optimize import minimize

class KantorovichProjectSolver:
    def __init__(self):
        self.results_log = []

    def solve_allocation(self, data, problem_type='A'):
        # 1. Extragere Date
        prob_id = data.get('id', data.get('name', 'Unknown'))
        raw_prod = data.get('productivity') or data.get('productivity_matrix')
        prod = np.array(raw_prod)
        rows, cols = prod.shape

        # Gestionare Capacitate Masini (ex: 3 freze = capacitate 3.0)
        machine_capacities = [1.0] * rows
        if 'machines' in data and isinstance(data['machines'], list):
            for idx, m in enumerate(data['machines']):
                if isinstance(m, dict) and 'count' in m and idx < rows:
                    machine_capacities[idx] = float(m['count'])

        # 2. Modelare PuLP
        model = pulp.LpProblem(f"Alloc_{prob_id}", pulp.LpMaximize)
        Z = pulp.LpVariable("Z", lowBound=0)
        h = pulp.LpVariable.dicts("h", ((i, j) for i in range(rows) for j in range(cols)), lowBound=0)

        model += Z

        # Constrangere: Capacitate Masini
        for i in range(rows):
            model += pulp.lpSum([h[(i, j)] for j in range(cols)]) == machine_capacities[i]

        # Constrangere: Balanta Productie
        for j in range(cols):
            model += pulp.lpSum([prod[i][j] * h[(i, j)] for i in range(rows)]) == Z

        # Constrangere: Resurse (Tip B)
        if problem_type == 'B':
            cons = data.get('resource_consumption') or data.get('energy_consumption')
            limit = data.get('max_resource') or data.get('max_energy')
            if cons and limit:
                try:
                    # Sum(consum_specific * timp_alocat) <= Limita
                    model += pulp.lpSum([cons[i][j] * h[(i, j)] for i in range(rows) for j in range(cols)]) <= limit
                except:
                    pass

        # 3. Rezolvare
        start = time.time()
        model.solve(pulp.PULP_CBC_CMD(msg=False))

        # Pregatire date pentru vizualizare
        allocation_matrix = np.zeros((rows, cols))
        for i in range(rows):
            for j in range(cols):
                allocation_matrix[i][j] = pulp.value(h[(i,j)])

        return {
            "id": prob_id,
            "val": pulp.value(Z),
            "time": (time.time() - start) * 1000,
            "allocation": allocation_matrix,
            "machines": data.get('machine_names', [f"M{i+1}" for i in range(rows)]),
            "parts": data.get('part_names', [f"P{j+1}" for j in range(cols)])
        }

    def plot_allocation(self, result):
        plt.figure(figsize=(10, 6))
        sns.heatmap(result['allocation'], annot=True, cmap="YlGnBu", fmt=".2f",
                    xticklabels=result['parts'], yticklabels=result['machines'])
        plt.title(f"Alocarea Optima: {result['id']} (Z = {result['val']:.2f})", fontsize=14)
        plt.xlabel("Piese (Produse)")
        plt.ylabel("Masini (Resurse)")
        plt.show()


## 4. Studiu de Caz: Problema Originală (Kantorovich, 1939)

Să recreăm exemplul din carte. Avem:
* **3 Freze (Milling Machines):** Rapide la Piesa 2, lente la Piesa 1.
* **3 Strunguri (Lathe Machines):** Echilibrate.
* **1 Automat:** Foarte rapid la Piesa 2.

Productivitatea este dată în piese/oră.

In [ ]:
# Definim datele problemei istorice
problem_kantorovich = {
    "id": "Kantorovich_Problem_A",
    "name": "Alocare Masini (1939)",
    "machines": [
      {"name": "Freze", "count": 3},
      {"name": "Strunguri", "count": 3},
      {"name": "Automate", "count": 1}
    ],
    "machine_names": ["Freze (3 buc)", "Strunguri (3 buc)", "Automat (1 buc)"],
    "part_names": ["Piesa 1", "Piesa 2"],
    # Productivitate [Masina][Piesa]
    "productivity": [
      [10, 20],  # Freze: fac 10 piese1 sau 20 piese2
      [20, 30],  # Strunguri
      [30, 80]   # Automat: foarte eficient la piesa 2
    ]
}

# Initializam solverul
solver = KantorovichProjectSolver()

# Rezolvam
rezultat = solver.solve_allocation(problem_kantorovich, problem_type='A')

print(f"\n--- REZULTATE ---")
print(f"Productia Maxima (Z): {rezultat['val']:.4f} seturi complete")
print(f"Timp de calcul: {rezultat['time']:.2f} milisecunde")

### Analiza Vizuală a Soluției

Graficul de mai jos (Heatmap) ne arată strategia optimă:
* Culorile închise indică o concentrare mare de resurse.
* Observați cum solverul "specializează" mașinile. De exemplu, Automatul ar trebui să lucreze aproape exclusiv la Piesa 2, unde are avantaj comparativ maxim.

In [ ]:
solver.plot_allocation(rezultat)

### Discuție: 86.00 vs 86.67

În cartea sa, Kantorovich ajunge la soluția de **86** de seturi. Algoritmul nostru modern arată **86.6667**.

**De ce apare diferența?**
Algoritmul Simplex lucrează cu numere reale continue (Programare Liniară). Matematic, optimul este $260/3 \approx 86.67$. Totuși, în realitate, nu poți livra 0.67 dintr-un set. Kantorovich, calculând manual sau folosind metode numerice timpurii, a rotunjit la cel mai apropiat număr întreg fezabil.

Acest lucru demonstrează acuratețea superioară a uneltelor digitale moderne.

## 5. Avansat: Problema cu Constrângeri Multiple (Tip B)

În realitate, nu doar timpul mașinii este limitat. Putem avea limite de energie electrică, materie primă sau forță de muncă.

Să modificăm problema: Adăugăm un consum de energie. Mașinile rapide consumă mai mult curent.
* Avem un buget de **150 kWh**.
* Frezele consumă 5 kWh/oră, Automatul consumă 15 kWh/oră.

In [ ]:
problem_with_energy = {
    "id": "Problem_B_Energy_Constraint",
    "machines": [
      {"name": "Freze", "count": 3},
      {"name": "Strunguri", "count": 3},
      {"name": "Automate", "count": 1}
    ],
    "productivity": [
      [10, 20],
      [20, 30],
      [30, 80]
    ],
    # Consum energie [kWh per unitate de timp alocata]
    "resource_consumption": [
        [5, 5],    # Frezele consuma putin
        [8, 8],    # Strungurile mediu
        [15, 20]   # Automatul consuma mult, mai ales la piesa 2
    ],
    "max_resource": 150, # Limita totala de energie
    "machine_names": ["Freze", "Strunguri", "Automat"],
    "part_names": ["Piesa 1", "Piesa 2"]
}

rezultat_B = solver.solve_allocation(problem_with_energy, problem_type='B')

print(f"Productia cu limita de energie: {rezultat_B['val']:.4f} seturi")
solver.plot_allocation(rezultat_B)

## 6. Concluzii

1. **Eficiență:** Programarea liniară permite găsirea soluției optime în milisecunde, o sarcină care ar dura ore întregi manual.
2. **Specializare:** Matematica ne arată că eficiența maximă se obține prin specializarea mașinilor (avantaj comparativ), nu prin a pune fiecare mașină să facă "câte puțin din toate".
3. **Scalabilitate:** Același cod poate rezolva o problemă cu 1000 de mașini și 500 de tipuri de piese fără modificări majore.